In [1]:
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


# =========================================================
# 224-226p: Scaled Dot-Product Attention
#  (あ) = q, k.transpose(-2, -1)   ← ここは -1 が正（1 だと次元不一致で動きません）
# =========================================================
class ScaledDotProductAttention(nn.Module):
    def __init__(self, scale_factor=1.0):
        super().__init__()
        self.scale_factor = scale_factor

    def forward(self, q, k, v):
        dk = k.size(-1)

        # スケールされた内積を計算
        # scores = torch.matmul((あ)) / torch.sqrt(dk * self.scale_factor)
        # ↑本の書き方に近い形にしつつ、PyTorchで素直に動くように書くと↓
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(dk * self.scale_factor)

        attention = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention, v)
        return output


# =========================================================
# 224-226p: Multihead Attention
#  (い) = self.attention(q, k, v)
# =========================================================
class MultiheadAttention(nn.Module):
    def __init__(self, num_heads, input_size, head_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size

        # Query, Key, Value の線形変換
        self.q_linear = nn.Linear(input_size, num_heads * head_size)
        self.k_linear = nn.Linear(input_size, num_heads * head_size)
        self.v_linear = nn.Linear(input_size, num_heads * head_size)

        self.attention = ScaledDotProductAttention(scale_factor=1.0)
        self.fc = nn.Linear(num_heads * head_size, input_size)

    def forward(self, q, k, v):
        bs = q.size(0)

        # 線形変換 → (bs, heads, len, head_size)
        q = self.q_linear(q).view(bs, -1, self.num_heads, self.head_size).transpose(1, 2)
        k = self.k_linear(k).view(bs, -1, self.num_heads, self.head_size).transpose(1, 2)
        v = self.v_linear(v).view(bs, -1, self.num_heads, self.head_size).transpose(1, 2)

        # Scaled Dot-Product Attention
        attention_scores = self.attention(q, k, v)  # (い)

        # heads を戻して結合
        attention = attention_scores.transpose(1, 2).contiguous().view(
            bs, -1, self.num_heads * self.head_size
        )
        output = self.fc(attention)
        return output


# =========================================================
# 224-226p: Positional Encoding（batch_first対応）
# =========================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # batch_first: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (N, T, d_model)
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


# =========================================================
# 226p: Transformer Block
# =========================================================
class TransformerBlock(nn.Module):
    def __init__(self, input_size, num_heads, head_size, dropout_rate=0.1):
        super().__init__()
        self.multihead_attention = MultiheadAttention(num_heads, input_size, head_size)

        self.layer_norm1 = nn.LayerNorm(input_size)
        self.dropout1 = nn.Dropout(dropout_rate)

        self.feedforward = nn.Sequential(
            nn.Linear(input_size, 4 * input_size),
            nn.ReLU(),
            nn.Linear(4 * input_size, input_size),
        )

        self.layer_norm2 = nn.LayerNorm(input_size)
        self.dropout2 = nn.Dropout(dropout_rate)

    def forward(self, x):
        attn_output = self.multihead_attention(x, x, x)  # self-attention
        x = self.layer_norm1(x + self.dropout1(attn_output))

        ff_output = self.feedforward(x)
        x = self.layer_norm2(x + self.dropout2(ff_output))

        return x


# =========================================================
# おもちゃタスク：ポインタ分類
# - 入力: [content..., POINTER(pos)]  ※最後のトークンが「何番目を見ろ」
# - ラベル: 指定位置の content_token % num_classes
# - これなら "1文→1ラベル" の分類なので、学習・評価ループを流用できる
# =========================================================
class PointerClassificationDataset(Dataset):
    def __init__(self, n_samples, seq_len, content_vocab_size, num_classes, seed=42):
        self.n_samples = n_samples
        self.seq_len = seq_len
        self.content_vocab_size = content_vocab_size
        self.num_classes = num_classes
        self.rng = random.Random(seed)

        # token id 設計
        self.pad_idx = 0
        self.pointer_base = content_vocab_size + 1  # POINTER(0) のid

        # vocab_size = [PAD] + content + pointer_tokens
        self.vocab_size = 1 + content_vocab_size + (seq_len - 1)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        T = self.seq_len
        content_len = T - 1

        # content tokens: 1..content_vocab_size
        content = [self.rng.randint(1, self.content_vocab_size) for _ in range(content_len)]

        # pointer position: 0..content_len-1
        pos = self.rng.randint(0, content_len - 1)
        pointer_token = self.pointer_base + pos

        tokens = content + [pointer_token]  # 長さ T
        label = content[pos] % self.num_classes

        tokens = torch.tensor(tokens, dtype=torch.long)
        length = torch.tensor(T, dtype=torch.long)
        label = torch.tensor(label, dtype=torch.long)
        return tokens, length, label


def collate_fixed(batch):
    # 全サンプル同じ長さなので stack でOK
    tokens = torch.stack([b[0] for b in batch], dim=0)   # (N, T)
    lengths = torch.stack([b[1] for b in batch], dim=0)  # (N,)
    labels = torch.stack([b[2] for b in batch], dim=0)   # (N,)
    return tokens, lengths, labels


# =========================================================
# これまでと同じ interface の Transformer 分類モデル
# forward(tokens, lengths) -> logits (N, C)
# =========================================================
class NetTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        pad_idx,
        embed_dim=64,
        num_classes=4,
        num_layers=2,
        num_heads=4,
        head_size=16,        # num_heads * head_size == embed_dim がわかりやすい
        dropout=0.1,
        max_len=5000,
    ):
        super().__init__()
        assert num_heads * head_size == embed_dim, "embed_dim は num_heads*head_size に合わせると分かりやすいです"

        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.posenc = PositionalEncoding(embed_dim, dropout=dropout, max_len=max_len)

        self.blocks = nn.ModuleList(
            [TransformerBlock(embed_dim, num_heads, head_size, dropout_rate=dropout) for _ in range(num_layers)]
        )

        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, tokens, lengths):
        # tokens: (N, T)
        x = self.embedding(tokens)  # (N, T, D)
        x = self.posenc(x)

        for blk in self.blocks:
            x = blk(x)

        # 最後のトークン（ポインタ位置）は常に lengths-1（固定長ならT-1）
        idx = (lengths - 1).clamp(min=0)  # (N,)
        last = x[torch.arange(x.size(0), device=x.device), idx]  # (N, D)

        logits = self.fc(last)  # (N, C)
        return logits


# =========================================================
# あなたの「学習 + 評価」ループ（ほぼそのまま）
# =========================================================
def main(NetClass, vocab_size, pad_idx, trainloader, testloader, epochs=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    net = NetClass(vocab_size=vocab_size, pad_idx=pad_idx).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=0.05, momentum=0.9)

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0

        for i, (tokens, lengths, labels) in enumerate(trainloader, 0):
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = net(tokens, lengths)  # (N, C)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if i % 100 == 99:
                print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}")
                running_loss = 0.0

    print("Finished Training")

    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, lengths, labels in testloader:
            tokens, lengths, labels = tokens.to(device), lengths.to(device), labels.to(device)
            outputs = net(tokens, lengths)
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {100.0 * correct / total:.2f}%")
    return net


# =========================================================
# 実行例
# =========================================================
seq_len = 12
content_vocab_size = 30
num_classes = 4

train_ds = PointerClassificationDataset(
    n_samples=8000,
    seq_len=seq_len,
    content_vocab_size=content_vocab_size,
    num_classes=num_classes,
    seed=0
)
test_ds = PointerClassificationDataset(
    n_samples=2000,
    seq_len=seq_len,
    content_vocab_size=content_vocab_size,
    num_classes=num_classes,
    seed=1
)

batch_size = 64
trainloader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fixed)
testloader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fixed)

# Transformerで回す（学習・評価ループは同じ）
net = main(NetTransformerClassifier, train_ds.vocab_size, train_ds.pad_idx, trainloader, testloader, epochs=5)


[1,   100] loss: 1.464
[2,   100] loss: 1.354
[3,   100] loss: 1.060
[4,   100] loss: 0.707
[5,   100] loss: 0.509
Finished Training
Test Accuracy: 97.60%
